# HW1 — Logic, SAT/SMT, and bounded reachability

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/hw1-logic-sat-smt.ipynb)

**CS 3892 / 5892 · out Thu Sep 10 · due Thu Sep 24, 11:59 p.m. US Central**

Starters for all four parts, running in a browser with nothing installed.

**These are starters, not solutions.** Everything complete here is worked on a
*different* instance from the one the homework asks about. The files with TODOs
are the ones that are yours:

| | |
|---|---|
| `p1_warmup_starter.py` | two worked, **five statements are yours** |
| `p3_reachability_starter.py` | **three TODOs** — the machine from the handout |

Everything else runs as-is and is there to be scaled up, timed, and copied from.

> Editing here does **not** change the repo. When you have something you want to
> keep, save it into your own copy — Colab's *File → Save a copy in Drive*, or
> work in a clone.

## Setup

Run once. Clones the repo on Colab, no-ops anywhere it already exists.

In [ ]:
# --- Setup: find the repo (clone on Colab), install Z3, define helpers -------
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
HW       = "hw1-logic-sat-smt"

def _find_repo():
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "homework" / HW).is_dir():
            return p
    dest = (pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir()
            else pathlib.Path.cwd() / "cs3892-examples")
    if not (dest / "homework" / HW).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

try:
    import z3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"], check=True)
    import z3
print("Z3", z3.get_version_string())

PYD = ROOT / "homework" / HW / "python"
SM  = ROOT / "homework" / HW / "smt2"

def show(path):
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 58 - len(path.name)))
    print(path.read_text().rstrip()); print()

def run(path, show_source=False):
    """Run one starter and stream its output. Raises if it fails.

    Python starters import their siblings (p2_timing pulls in p2_nqueens), so
    each runs from its own directory. .smt2 goes through the repo's runner,
    which checks the file's own `; EXPECT:` line -- the z3-solver wheel ships
    no `z3` CLI, so there is nothing to shell out to on Colab.
    """
    path = pathlib.Path(path)
    if show_source:
        show(path)
    if path.suffix == ".smt2":
        cmd, cwd = [sys.executable, str(ROOT / "scripts" / "run_smt2.py"), str(path)], ROOT
    else:
        cmd, cwd = [sys.executable, path.name], path.parent
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(cwd))
    print(r.stdout.rstrip())
    if r.stderr.strip():
        print(r.stderr.rstrip(), file=sys.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{path.name} failed")

print("ready — helpers: show(path), run(path)")

## Part 1 — Logic warm-up · 20 points

Translate five English statements into logic, **decide satisfiability and validity by hand first**,
then confirm with Z3.

The starter works two examples: *modus ponens* (valid) and *affirming the consequent* (satisfiable
but not valid — and the counterexample it prints is the explanation). Then five slots are yours.

Remember why you need both questions: a solver only answers *satisfiable*. Validity comes from
**φ is valid ⟺ ¬φ is unsatisfiable**.

In [ ]:
run(PYD / "p1_warmup_starter.py")

Read it before you edit it:

In [ ]:
show(PYD / "p1_warmup_starter.py")

## Part 2 — Encoding · 30 points

Two complete encoders to choose between and scale up. They are deliberately different shapes:
pigeonhole is **boolean** and its clause count is quadratic in the number of pigeons; n-queens is
**integer** and far smaller. Which one hits the wall first, and why, is most of what part 2 is
asking.

Pigeonhole is also the classic hard case: any resolution proof of *n+1 into n* is exponential in n.
That wall is a theorem, not Z3 being slow.

In [ ]:
run(PYD / "p2_pigeonhole.py")
run(PYD / "p2_nqueens.py")
run(SM  / "p2_pigeonhole_5_into_4.smt2")

### The timing table

This produces the table part 2 asks for. **The ranges are deliberately small so it finishes here —
widen them for your report** until each one stops finishing, then say where the wall is and what
about the encoding put it there.

In [ ]:
run(PYD / "p2_timing.py")

## Part 3 — Bounded reachability · 30 points

The method, in four lines: unroll *k* steps into *k+1* variables, constrain each transition, assert
the target appears somewhere, and ask. `sat` means reachable — **and the model is the trace**.

The demo works a toy: x starts at 0, each step adds 1 or 2, target 7. The threshold is k = 4.
Below it the answer is `unsat`, and that is arithmetic, not the solver.

**Your machine is a different one — read the handout.** Copy the method, not the number.

In [ ]:
run(PYD / "p3_reachability_demo.py")

Same unrolling written by hand in SMT-LIB, either side of the threshold:

In [ ]:
run(SM / "p3_reach7_k3.smt2")
run(SM / "p3_reach7_k4.smt2")

And the shell for yours — three TODOs:

In [ ]:
run(PYD / "p3_reachability_starter.py")
show(PYD / "p3_reachability_starter.py")

## Part 4 — AI component · 20 points

Ask an LLM for the SMT-LIB encoding of part 3, then answer the only question that matters:
**how did you know whether it was right?**

This is harder than it sounds. `unsat` is what a *correct* proof of unreachability returns — and it
is also what a contradictory, over-constrained, or empty encoding returns. You cannot tell them
apart by reading.

The starter runs three checks that *can* tell them apart, against an encoding sabotaged with a
plausible-looking one-line typo. Run all three against whatever the model gives you; whichever one
fails is your answer.

In [ ]:
run(PYD / "p4_how_do_you_know.py")

## Scratch

Yours. Nothing here is saved back to the repo.

In [ ]:
from z3 import *

# e.g. start on part 1 here, then move it into the starter file.
p, q = Bools("p q")
phi = Implies(And(Implies(p, q), q), p)      # affirming the consequent
s = Solver(); s.add(Not(phi))
print(s.check())
if s.check() == sat:
    print("counterexample:", s.model())

## Submitting

Through Brightspace → Assignments → *HW1 — Logic; SAT-SMT; bounded reachability (Z3)*.

1. Your `.py` and/or `.smt2` files, runnable as-is
2. A short report (~2 pages) with the answers, the timing table, and the part-4 write-up
3. Your AI-use disclosure

Four late days for the semester, at most two on any one homework. Beyond that, 20% per day.

Source: [`homework/hw1-logic-sat-smt/`](https://github.com/ttj/cs3892-examples/tree/main/homework/hw1-logic-sat-smt) ·
[handout](https://github.com/ttj/cs3892-fmstai-fall2026/blob/main/assignments/hw1.md)